
# Propensity Score Matching (PSM): A Mini-Tutorial

This notebook walks you through a practical, tutorial-style workflow for estimating a causal effect when treatment assignment is **confounded** by observed covariates.

We proceed in three stages:
1. **Explain the problem** and why naïve comparisons are biased.  
2. **Show the "normal" solution with PSM**—estimate propensities, match, and estimate the treatment effect with balance checks.  
3. **Use CAIS to automate the workflow** end-to-end and **explain the output** it produces.



## 1) Problem setup (what are we trying to estimate?)

We want the **causal effect of a binary treatment \(T \in \{0,1\}\)** on an outcome \(Y\), e.g., the effect of a program on child outcomes or a marketing intervention on spend.  
The challenge is that treatment is **not randomized**: treated and control units differ systematically in observed covariates \(X\) (e.g., age, income, risk factors), creating **confounding**.

A naïve difference in means,
\[
\mathbb{E}[Y\mid T{=}1] - \mathbb{E}[Y\mid T{=}0],
\]
is generally **biased** for the Average Treatment effect on the Treated (ATT) because \(X\) shifts both treatment propensity and the outcome.

**Key identification idea:** If we assume **unconfoundedness given covariates** (a.k.a. selection on observables) and **overlap**, then conditioning on the **propensity score** \(e(x)=\Pr(T{=}1\mid X{=}x)\) balances the covariates between treated and control, enabling unbiased ATT estimation.



### Data we will use

We'll use the IHDP dataset variant in this repo (e.g., `data/ihdp_0.csv`), which is commonly used to illustrate treatment effect estimation with selection on observables. It contains:
- `t`: treatment indicator (1/0)
- `y_factual`: the observed outcome
- `x*`: pre-treatment covariates

*(If your local file uses slightly different column names, adjust the code below accordingly.)*


In [1]:

import pandas as pd

# Load the IHDP sample provided with this project
df = pd.read_csv("data/ihdp_0.csv")

# Quick peek
df.head()


,treatment,y,x1,x2,x3,x4,x5,x6,x7,x8,...,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25
0,1,5.599916,-0.528603,-0.343455,1.128554,0.161703,-0.316603,1.295216,1,0,...,1,1,1,1,0,0,0,0,0,0
1,0,6.875856,-1.736945,-1.802002,0.383828,2.244320,-0.629189,1.295216,0,0,...,1,1,1,1,0,0,0,0,0,0
2,0,2.996273,-0.807451,-0.202946,-0.360898,-0.879606,0.808706,-0.526556,0,0,...,1,0,1,1,0,0,0,0,0,0
3,0,1.366206,0.390083,0.596582,-1.850350,-0.879606,-0.004017,-0.857787,0,0,...,1,0,1,1,0,0,0,0,0,0
4,0,1.963538,-1.045229,-0.602710,0.011465,0.161703,0.683672,-0.360940,1,0,...,1,1,1,1,0,0,0,0,0,0



## 2) The "normal" solution: Propensity Score Matching (PSM)

**Goal:** Estimate the **ATT** (effect for treated units) by pairing each treated unit with one (or more) similar control unit(s) matched on the **propensity score**.

**Steps:**
1. Fit a **propensity model**: logistic regression \(T \sim X\).
2. Compute **propensity scores** \(\hat e(X)\) for all units.
3. **Match** treated to control (e.g., nearest-neighbor with caliper) using \(\hat e\).
4. Check **covariate balance** after matching (standardized mean differences, plots).
5. Estimate **ATT** on the matched sample and report uncertainty.


In [3]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Identify columns
t_col = "t"
y_col = "y_factual"
X_cols = [c for c in df.columns if c not in [t_col, y_col]]

# 1) Fit propensity model
logit = LogisticRegression(max_iter=2000, solver="lbfgs")
logit.fit(df[X_cols], df[t_col])
df["ps"] = logit.predict_proba(df[X_cols])[:, 1]

# 2) Split treated/control
treated = df[df[t_col] == 1].copy()
control = df[df[t_col] == 0].copy()

# 3) Nearest-neighbor matching on propensity score (1:1, with simple replacement)
nbrs = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(control[["ps"]])
dist, idx = nbrs.kneighbors(treated[["ps"]])
treated["match_idx"] = idx.flatten()
treated["match_dist"] = dist.flatten()

# Build matched control sample
matched_controls = control.iloc[treated["match_idx"].values].copy()
matched_controls.index = treated.index  # align indices for subtraction

# 4) ATT on matched pairs
att = (treated[y_col] - matched_controls[y_col]).mean()

# A simple bootstrap for CI (fast but illustrative)
rng = np.random.default_rng(0)
B = 500
boot_ests = []
idxs = treated.index.to_numpy()
for _ in range(B):
    b_idx = rng.choice(idxs, size=len(idxs), replace=True)
    boot_ests.append((treated.loc[b_idx, y_col] - matched_controls.loc[b_idx, y_col]).mean())
boot_ests = np.array(boot_ests)
ci_low, ci_high = np.percentile(boot_ests, [2.5, 97.5])

att, (ci_low, ci_high)


KeyError: 't'


> **Interpretation (PSM):**  
> `att` is the estimated causal effect on the treated; the bootstrap interval provides a rough 95% CI. In production, you should add **balance diagnostics** (e.g., standardized mean differences pre/post match) and consider calipers, matching without replacement, or doubly robust estimators.



## 3) Automating with CAIS

Instead of hand-rolling the PSM pipeline, we can hand the problem to **CAIS** (the Causal Agent), which can:
- Parse your causal **query**,
- Inspect the **dataset** and variables,
- Choose and run an appropriate **estimator** (e.g., PSM, IPW, DR, or meta-learners),
- Produce **diagnostics** and an **effect estimate** with uncertainty.

We just provide a minimal structured query and dataset information.


In [ ]:

# If you have CAIS installed and available in this environment:
# from causal_agent import run_causal_analysis

query = "What is the ATT of the treatment (t) on the outcome (y_factual) in the IHDP sample, adjusting for observed covariates?"
dataset_path = "data/ihdp_0.csv"
dataset_description = """
Dataset: IHDP variant sample.
Columns:
- t: treatment indicator (1/0)
- y_factual: observed outcome
- Remaining columns: pre-treatment covariates X.
Goal: Estimate the ATT of t on y_factual under selection-on-observables.
"""

# result = run_causal_analysis(
#     query=query,
#     dataset_path=dataset_path,
#     dataset_description=dataset_description,
# )

# For illustration in this static notebook (if CAIS isn't installed), we'll mock a plausible structure:
result = {
    "estimator": "PSM (1:1 NN match on propensity)",
    "effect": float(att),
    "ci": [float(ci_low), float(ci_high)],
    "n_treated": int((df["t"]==1).sum()),
    "n_control": int((df["t"]==0).sum()),
    "diagnostics": {
        "overlap": True,
        "balance_after_matching": "improved",
        "notes": "Standardized mean differences reduced on most covariates."
    }
}
result



## 4) Explaining CAIS Output

A typical CAIS response includes:

- **`estimator`**: Which causal estimator it selected (e.g., PSM, IPW, DR, S-learner/T-learner).  
- **`effect`** and **`ci`**: Point estimate and 95% confidence interval for the target estimand (here, **ATT**).  
- **Sample sizes**: Number of treated and control units used.  
- **Diagnostics**: Notes on overlap/positivity and covariate balance before/after adjustment.

**How to read it:** If the CI excludes 0, you have evidence of a non-zero treatment effect under the modeling and identification assumptions. Always validate **overlap** and **balance**; if overlap is weak, consider trimming, calipers, or alternate estimators.



### Takeaways
- PSM is a classical, transparent approach for handling confounding under selection on observables.  
- CAIS can **automate** estimator choice, execution, and diagnostics, reducing boilerplate and helping you focus on **interpretation** and **assumptions**.  
- Regardless of automation, always review **diagnostics** and **sensitivity** to modeling choices.


In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from dowhy import CausalModel

# Load the dataset
data_path = "data/ihdp_0.csv"
df = pd.read_csv(data_path)

# Identify the treatment and outcome variables
treatment = "treatment"
outcome = "y"
covariates = df.columns[2:].tolist()  # all columns except treatment and outcome

# Create a causal model
model = CausalModel(
    data=df, treatment=treatment, outcome=outcome, common_causes=covariates
)

# Identify the causal effect
identified_estimand = model.identify_effect()
effect_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
    target_units="ate",
)

# Print the estimated effect
print("Estimated Effect of Home Visits on Cognitive Scores:", effect_estimate)


propensity_score_matching
Estimated Effect of Home Visits on Cognitive Scores: *** Causal Estimate ***

## Identified estimand
Estimand type: nonparametric-ate

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                                                         ↪
────────────(E[y|x7,x12,x9,x16,x23,x20,x22,x1,x10,x11,x5,x17,x3,x24,x4,x2,x8,x ↪
d[treatment]                                                                   ↪

↪                                
↪ 6,x25,x13,x18,x19,x15,x14,x21])
↪                                
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→y then P(y|treatment,x7,x12,x9,x16,x23,x20,x22,x1,x10,x11,x5,x17,x3,x24,x4,x2,x8,x6,x25,x13,x18,x19,x15,x14,x21,U) = P(y|treatment,x7,x12,x9,x16,x23,x20,x22,x1,x10,x11,x5,x17,x3,x24,x4,x2,x8,x6,x25,x13,x18,x19,x15,x14,x21)

## Realized estimand
b: y~treatment+x7+x12+x9+x16+x23+x20+x22+x1+x10+x11+x5+x17+x3+x24+x4+x2+x8+x6+x25+x13+x18+x19+x15+x14+x21
Target units: at

/home/sam/.local/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [4]:
from auto_causal import run_causal_analysis


dataset_description = """
The CSV file ihdp_1.csv contains data obtained from the Infant Health and Development Program (IHDP). The randomized study is designed to evaluate the effect of home visit from specialist doctors on the cognitive test scores of premature infants. The confounders x (x1-x25) correspond to collected measurements of the children and their mothers, including measurements on the child (birth weight, head circumference, weeks born preterm, birth order, first born, neonatal health index, sex, twin status), as well as behaviors engaged in during the pregnancy (smoked cigarettes, drank alcohol, took drugs) and measurements on the mother at the time she gave birth (age, marital status, educational attainment, whether she worked during pregnancy, whether she received prenatal care) and the site (8 total) in which the family resided at the start of the intervention. There are 6 continuous covariates and 19 binary covariates.
"""

# Run causal analysis with a simple question
result = run_causal_analysis(
    query="Do home visits from specialist doctors lead to an improvement in cognitive scores?",
    dataset_path="data/ihdp_1.csv",
    dataset_description=dataset_description
)
print(result)
print(f"Causal effect: {result['results']['results']['effect_estimate']}")
print(f"Method used: {result['results']['results']['method_used']}")

2025-09-02 19:56:13,422 - INFO - Starting causal analysis run...
2025-09-02 19:56:13,424 - INFO - Initializing LLM client: Provider='together', Model='deepseek-ai/DeepSeek-V3'
2025-09-02 19:56:13,485 - INFO - Constructed input for agent: 
My question is: Do home visits from specialist doctors lead to an improvement in cognitive scores?
The dataset is located at: data/ihdp_1.csv
Dataset Description: 
The CSV file ihdp_1.csv contains data obtained from the Infant Health and Development Program (IHDP). The randomized study is designed to evaluate the effect of home visit from specialist doctors on the cognitive test scores of premature infants. The confounders x (x1-x25) correspond to collected measurements of the children and their mothers, including measurements on the child (birth weight, head circumference, weeks born preterm, birth order, first born, neonatal health index, sex, twin status), as well as behaviors engaged in during the pregnancy (smoked cigarettes, drank alcohol, took 

[HumanMessage(content='\nAnalyze the following causal query **strictly in the context of the provided dataset information (if available)**. Identify the query type, key variables (mapping query terms to actual column names when possible), constraints, and any explicitly mentioned dataset path.\n\nUser Query: "Do home visits from specialist doctors lead to an improvement in cognitive scores?"\n\nNo dataset context provided.\n\n# Add specific guidance for query types\nGuidance for Identifying Query Type:\n- EFFECT_ESTIMATION: Look for keywords like \'effect\', \'impact\', \'influence\', \'cause\', \'affect\', \'consequence\'. Also consider questions asking "how does X affect Y?" or comparing outcomes between groups based on an intervention.\n- COUNTERFACTUAL: Look for hypothetical scenarios, often using phrases like \'what if\', \'if X had been\', \'would Y have changed\', \'imagine if\', \'counterfactual\'.\n- CORRELATION: Look for keywords like \'correlation\', \'association\', \'relat

2025-09-02 19:56:16,779 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:56:16,781 - WARNING - LLM identified time variable 'null' not found in columns. Setting to None.
2025-09-02 19:56:16,781 - WARNING - LLM identified unit variable 'null' not found in columns. Setting to None.
2025-09-02 19:56:16,782 - INFO - LLM identified time='None', unit='None'
2025-09-02 19:56:16,783 - INFO - Not panel data: Missing either time or unit variable for panel structure.
2025-09-02 19:56:16,784 - INFO - Using LLM to identify potential treatment and outcome variables
2025-09-02 19:56:17,872 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:56:17,875 - INFO - LLM identified 1 treatments and 1 outcomes
2025-09-02 19:56:17,876 - INFO - Using LLM to identify potential instrumental variables
2025-09-02 19:56:20,081 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 

--------------------------
Validation result: {'valid': True, 'concerns': [], 'alternative_suggestions': [], 'recommended_method': 'linear_regression', 'assumptions': ['linear relationship between treatment, covariates, and outcome', 'no unmeasured confounders (if observational)', 'correct model specification', 'homoscedasticity of errors', 'normally distributed errors (for inference)']}
--------------------------


2025-09-02 19:56:38,681 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:56:38,685 - WARNING - LLM parameter identification did not yield usable parameters. Falling back to regex.
2025-09-02 19:56:38,708 - INFO - Method execution successful. Effect estimate: None
2025-09-02 19:56:38,711 - INFO - Running explainer_tool with direct arguments...
2025-09-02 19:56:38,713 - INFO - Initializing LLM client: Provider='together', Model='deepseek-ai/DeepSeek-V3'
2025-09-02 19:56:38,764 - INFO - explanation_generator_tool finished successfully.
2025-09-02 19:56:38,766 - INFO - {'query': 'Do home visits from specialist doctors lead to an improvement in cognitive scores?', 'method': 'linear_regression', 'results': {'results': {'effect_estimate': None, 'confidence_interval': [None, None], 'standard_error': None, 'p_value': None, 'method_used': 'linear_regression', 'llm_assumption_check': None, 'raw_results': None, 'diagnostics': {}, 'refutation_

summary_dict: {'query': 'Do home visits from specialist doctors lead to an improvement in cognitive scores?', 'method_used': 'Linear Regression (OLS)', 'causal_effect': None, 'standard_error': None, 'confidence_interval': [None, None]}
CURRENT_OUTPUT_LOG_FILE: None
{'query': 'Do home visits from specialist doctors lead to an improvement in cognitive scores?', 'method': 'linear_regression', 'results': {'results': {'effect_estimate': None, 'confidence_interval': [None, None], 'standard_error': None, 'p_value': None, 'method_used': 'linear_regression', 'llm_assumption_check': None, 'raw_results': None, 'diagnostics': {}, 'refutation_results': None}, 'variables': {'treatment_variable': 'treatment', 'treatment_variable_type': 'binary', 'outcome_variable': 'y', 'instrument_variable': None, 'covariates': ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'], 'time_variable': None, 'group_variable': None, 'running_variable': None, 'cutoff_value': None, 'is_rct': True, 'treatment_refere

In [5]:
print(result)

{'query': 'Do home visits from specialist doctors lead to an improvement in cognitive scores?', 'method': 'linear_regression', 'results': {'results': {'effect_estimate': None, 'confidence_interval': [None, None], 'standard_error': None, 'p_value': None, 'method_used': 'linear_regression', 'llm_assumption_check': None, 'raw_results': None, 'diagnostics': {}, 'refutation_results': None}, 'variables': {'treatment_variable': 'treatment', 'treatment_variable_type': 'binary', 'outcome_variable': 'y', 'instrument_variable': None, 'covariates': ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'], 'time_variable': None, 'group_variable': None, 'running_variable': None, 'cutoff_value': None, 'is_rct': True, 'treatment_reference_level': '0', 'interaction_term_suggested': False, 'interaction_variable_candidate': None}, 'dataset_analysis': {'dataset_info': {'num_rows': 747, 'num_columns': 27, 'file_path': 'data/ihdp_1.csv', 'file_name': 'ihdp_1.csv'}, 'columns': ['treatment', 'y', 'x1', '